In [81]:
# importing necessary python modules for pyspark execution
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,\
                            trim, upper, avg, round,\
                            sum as spark_sum, to_date, when, count,\
                            current_timestamp, regexp_replace, regexp,\
                            broadcast, round as spark_round, max as spark_max, mode,\
                            lit, desc, collect_list, min as spark_min, median as spark_median
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType, FloatType

In [ ]:
##################################################################################################################
###################################### SparkSession Creation function ############################################
########################## This helps to create sparksession #####################################################
##################################################################################################################

In [2]:
# Create helper static fucntion for creating spark session
@staticmethod
def create_spark_session():
    spark = SparkSession.builder\
                         .appName('Mini_Assignment1_Session')\
                         .getOrCreate()
    return spark

In [ ]:
##################################################################################################################
###################################### Metadata helper class #####################################################
########################## This helps to get the metadta of the dataframe ########################################
##################################################################################################################

In [24]:
# Lets create a helper class which provide metadata information of the dataframe
class extractMetadata:
    """
    This class will get dataframe as input and extract its metadata information
    """
    def __init__(self, df):
        """
        Initialize with dataframe --> used in getting the metadata
        Initialize with dataframe's row count and columns --> which provides the dataframe shape
        """
        self.df = df
        self.row_count = df.count()
        self.columns = df.columns
    # Below function will get the dataframe shape and return that as tuple (row_count, column_count)
    def _get_dataframe_shape(self):
        return (self.row_count, len(self.columns))
    # Below function will get the column names and store that in a python list
    def _get_column_names(self):
        return self.df.columns
    # Below function will get the column data types and store that in a python list
    def _get_column_types(self):
        return [datatype[1] for datatype in self.df.dtypes]
    # Below function will get the missing values count for each column and store that in a python list
    def _get_missing_column_counts(self):
        return [self.df.filter(col(column).isNull()).count() for column in self.columns]
    # Below function will get the missing values proportion for each column and store that in a python list
    def _get_missing_column_proportion(self):
        return [self.df.filter(col(column).isNull()).count()/self.row_count for column in self.columns]
    # Below function will get the distinct count of each column and store that in a python list
    def _get_unique_column_counts(self):
        return [self.df.select(col(column)).groupBy(col(column)).count().select(column).count() for column in self.columns]
    # Below function will get index column possibility and store that in a python list
    def _index_column_possiblity(self):
        return ["All Unique" if self.df.select(col(column)).groupBy(col(column)).count().select(column).count() == self.row_count else "Non Unique Column" for column in self.columns]
    # Below function will get the categorical columns top 3 mode value and store that in a python list
    def _get_categorical_mode_values(self):
        categorical_list = []
        for field in self.df.schema.fields:
            if field.dataType not in (DoubleType(),IntegerType()):
                top_3_rows = (
                self.df.groupBy(col(field.name))
                .count()
                .orderBy(desc('count'))
                .head(3)
                )
                top_3_values = [f'{row[0]}-{row[1]}' for row in top_3_rows]
                categorical_list.append(top_3_values)
            else:
                categorical_list.append(['Not a categorical column'])
        return categorical_list
    # Below function will get the numerical columns min, max, median value and store that in a python list
    def _get_numerical_distribution_values(self):
        numerical_list = []
        for field in self.df.schema.fields:
            if field.dataType in (DoubleType(),IntegerType()):
                dist_dict = ([
                            f'min - {self.df.select(spark_min(field.name)).collect()[0][0]}, median - {self.df.select(spark_median(field.name)).collect()[0][0]}, max - {self.df.select(spark_max(field.name)).collect()[0][0]}'
                            ])
                numerical_list.append(dist_dict)
            else:
                numerical_list.append(([f'{field.name}: Not a numerical column']))
        single_column_data = [(item,) for item in numerical_list]
        return single_column_data
    def extract_metadata(self, spark):
        metadata = list(zip(self._get_column_names(),
                            self._get_column_types(),
                            self._get_missing_column_counts(),
                            self._get_missing_column_proportion(),
                            self._get_unique_column_counts(),
                            self._index_column_possiblity(),
                            self._get_categorical_mode_values(),
                            self._get_numerical_distribution_values()
                           )
                       )
        columns = ["column_name",
                   "data_type",
                   "missing_column_count",
                   "missing_column_proportion(%)",
                   "unique_column_count",
                   "index_column_possibility",
                   "top_3_values",
                   "numerical_column_distribution"
                  ]
        return spark.createDataFrame(metadata,
                                    columns)
        
        

In [ ]:
##################################################################################################################
###################################### sparkFunctions function ###################################################
########################## This helps to read csv file, tsv file #################################################
########################## This helps to changing the datatype(standardizing the datatypes) ######################
########################## This helps to removing the spaces from the columns ####################################
########################## This helps to standardize null values #################################################
##################################################################################################################

In [4]:
class sparkFunctions:
    """
    This class 
        read CSV file, 
        standardize the columns,
        remove duplicates,
        remove unwanted columns,
        remove unwanted rows
    """
    def __init__(self, spark):
        self.spark = spark
    def read_csv_file(self, input_file, schema=None):
        """
        This function will read the data from comma seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def read_tsv_file(self, input_file, schema=None):
        """
        This function will read the data from tab seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, sep='\t', inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def remove_specific_strings_from_numeric(self, df, column_dict):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column_key,column_value in column_dict.items():
            standardized_df = standardized_df.withColumn(column_key,regexp_replace(col(column_key), column_value, ""))
        return standardized_df
    def remove_any_string_from_numeric(self, df, col_list):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             when(col(column).rlike("^[^0-9]+$"), None).otherwise(col(column))
                                           )
        return standardized_df
    def remove_spaces_from_string(self, df, col_list):
        """
        This function will remove the leading and trailing spaces from string columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             trim(col(column))
                                           )
        return standardized_df
    def standardize_null_values(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        standardized_df = standardized_df.fillna(column_dict)
        return standardized_df
    def standardize_datatypes(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        for column_name, column_type in column_dict.items():
            standardized_df = standardized_df.withColumn(column_name, col(column_name).cast(column_type))
        return standardized_df

In [5]:
# Create spark session
# Lets call the helper function to create the spark session
spark=create_spark_session()

In [6]:
# Let's read the csv file using the sparkFunction helper class
df = sparkFunctions(spark).read_csv_file('Data/Lung Cancer.csv')

In [ ]:
# Read the metadata using metadata helper class and keep this ready in the dataframe for future analysis
df_metadata = extractMetadata(df).extract_metadata(spark)

In [28]:
df_metadata.show(truncate=False)

+------------------+---------+--------------------+----------------------------+-------------------+------------------------+------------------------------------------------------------------+----------------------------------------------+
|column_name       |data_type|missing_column_count|missing_column_proportion(%)|unique_column_count|index_column_possibility|top_3_values                                                      |numerical_column_distribution                 |
+------------------+---------+--------------------+----------------------------+-------------------+------------------------+------------------------------------------------------------------+----------------------------------------------+
|id                |int      |0                   |0.0                         |890000             |All Unique              |[Not a categorical column]                                        |{[min - 1, median - 445000.5, max - 890000]}  |
|age               |double   |0         

In [ ]:
# Assignment questions:
# Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format. 
# Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. Then, return the average treatment duration for each treatment type.  
# Task 3: Write a function that returns the smoking_status group with the highest survival rate.  
# Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  
# Task 5: Write a function that filters patients who:  
# Are male  
# Diagnosed in Stage III or IV  
# Have a family history of cancer  
# Are current smokers  
# Have a BMI > 30  
# Survived 
# Return the average age and the percentage of these patients who had hypertension.

In [ ]:
###################################################
################# Task 1 started ##################
###################################################

In [31]:
# Task 1: Write a function that removes duplicate rows, ensures correct data types for numerical and date columns and converts all ‘yes’/ ‘no’ type fields into 1/0 format. 
def task1(df):
    ### clean row level duplicates ###
    df_task1 = df.dropDuplicates()
    ### Assumption ####
    ### No issues with string or date columns ###
    ### As per task 1, we need convert all yes/no to 1/0 format
    ### lets cast it(family_history) to integer after converting it to 1/0
    df_task1 = df_task1.withColumn('family_history', when(col('family_history') == 'Yes', 1)\
                                   .when(col('family_history') == 'No', 0)\
                                   .otherwise(col('family_history')))\
                        .withColumn('family_history', col("family_history").cast(IntegerType()))
    return df_task1    

In [32]:
# Calling task 1
df_final = task1(df)

In [ ]:
df_metadata_task1 = extractMetadata(df_final).extract_metadata(spark)

In [40]:
df_metadata_task1.filter(col('column_name') == 'family_history').show(truncate=False, vertical=True)

-RECORD 0-----------------------------------------------------------
 column_name                   | family_history                     
 data_type                     | int                                
 missing_column_count          | 0                                  
 missing_column_proportion(%)  | 0.0                                
 unique_column_count           | 2                                  
 index_column_possibility      | Non Unique Column                  
 top_3_values                  | [Not a categorical column]         
 numerical_column_distribution | {[min - 0, median - 0.0, max - 1]} 



In [ ]:
###################################################
################# Task 1 completed ################
###################################################

In [ ]:
###################################################
################# Task 2 started ##################
###################################################

In [69]:
# Task 2: Write a function that adds a new column, treatment_duration_days, which calculates the number of days between the diagnosis and the end of treatment. 
# Then, return the average treatment duration for each treatment type.  
def task2(df):
    ### Create new column called 'treatment_duration_days' ###
    df_task2 = df.withColumn('treatment_duration_days', (col('end_treatment_date') - col('diagnosis_date')).cast(IntegerType()))
    ### Get the average of treatment duration ###
    avg_treatment_duration_each_treatment = df_task2.groupBy('treatment_type').agg(avg(col('treatment_duration_days')).alias('average')).collect()
    ### Store the result in dictionary ###
    avg_treatment_duration = {row.treatment_type: row.average for row in avg_treatment_duration_each_treatment}
    return df_task2, avg_treatment_duration

In [70]:
# Calling task 2
df_final,avg_treatment_duration  = task2(df_final)

In [71]:
print(f"-"*45)
print(f"Mini Assignment 1 - Task 2")
print(f"-"*45)
print(f"Average treatment duration for each treatment type: {avg_treatment_duration}")

---------------------------------------------
Mini Assignment 1 - Task 2
---------------------------------------------
Average treatment duration for each treatment type: {'Radiation': 458.40320462900917, 'Chemotherapy': 458.39540091909953, 'Combined': 457.8152186120058, 'Surgery': 457.73744630723684}


In [ ]:
###################################################
################# Task 2 completed ################
###################################################

In [ ]:
###################################################
################# Task 3 started ##################
###################################################

In [105]:
# Task 3: Write a function that returns the smoking_status group with the highest survival rate.  
def task3(df):
    ### Survival Rate = (People in the group/total people)*100 ###
    task3 = df.groupBy('smoking_status').agg(round((spark_sum(col('survived'))/extractMetadata(df)._get_dataframe_shape()[0])*100,4).alias('survival_rate_by_smoking_status'))\
                                           .orderBy(desc(col('survival_rate_by_smoking_status'))).head(1)
    ### Store the result in dictionary ###
    highest_survival_rate = {row.smoking_status: row.survival_rate_by_smoking_status for row in task3}
    return highest_survival_rate

In [106]:
# Calling task 3
highest_survival_rate  = task3(df_final)
print(f"-"*45)
print(f"Mini Assignment 1 - Task 3")
print(f"-"*45)
print(f"Smoking_status group with the highest survival rate: {highest_survival_rate}")

---------------------------------------------
Mini Assignment 1 - Task 3
---------------------------------------------
Smoking_status group with the highest survival rate: {'Never Smoked': 5.529}


In [ ]:
###################################################
################# Task 3 completed ################
###################################################

In [ ]:
###################################################
################# Task 4 started ##################
###################################################

In [123]:
# Task 4: Write a function that returns the top three countries with the highest percentage of patients diagnosed in Stage IV.  
def task4(df):
    ### Survival Rate = (People in the group/total people)*100 ###
    task4 = df.filter(col('cancer_stage') == 'Stage IV')\
              .groupBy('country')\
              .agg(round((count(col('id'))/extractMetadata(df)._get_dataframe_shape()[0])*100,4)\
              .alias('count_percent')).orderBy(desc(col('count_percent'))).head(3)
    ### Store the result in dictionary ###
    highest_percentage_stage_iv_countries = [row for row in task4]
    return highest_percentage_stage_iv_countries

In [124]:
# Calling task 4
highest_percentage_stage_iv_countries  = task4(df_final)
print(f"-"*45)
print(f"Mini Assignment 1 - Task 4")
print(f"-"*45)
print(f"Top three countries with the highest percentage of patients diagnosed in Stage IV: {highest_percentage_stage_iv_countries}")

---------------------------------------------
Mini Assignment 1 - Task 4
---------------------------------------------
Top three countries with the highest percentage of patients diagnosed in Stage IV: [Row(country='Greece', count_percent=0.9471), Row(country='Croatia', count_percent=0.9467), Row(country='Malta', count_percent=0.9424)]


In [125]:
###################################################
################# Task 4 completed ################
###################################################

In [ ]:
###################################################
################# Task 5 started ##################
###################################################

In [144]:
# Task 5: Write a function that filters patients who:  
# Are male  
# Diagnosed in Stage III or IV  
# Have a family history of cancer  
# Are current smokers  
# Have a BMI > 30  
# Survived 
# Return the average age and the percentage of these patients who had hypertension. 
def task5(df):
    ### Survival Rate = (People in the group/total people)*100 ###
    task5 = df.filter(col('gender') == 'Male')\
              .filter((col('cancer_stage') == 'Stage IV') | (col('cancer_stage') == 'Stage III'))\
              .filter(col('family_history') == 1)\
              .filter(col('smoking_status') == 'Current Smoker')\
              .filter(col('bmi') > 30)\
              .filter(col('survived') == 1)\
              .filter(col('hypertension') == 1)\
              .agg(avg(col('age')).alias('average_age'),\
                   round((count(col('id'))/extractMetadata(df)._get_dataframe_shape()[0])*100,4).alias('count_percent')).collect()
    ### Store the result in dictionary ###
    task_5_answer = [row for row in task5]
    return task_5_answer

In [145]:
# Calling task 5
task_5_answer  = task5(df_final)
print(f"-"*45)
print(f"Mini Assignment 1 - Task 5")
print(f"-"*45)
print(f"Average age and patient percentage: {task_5_answer}")

---------------------------------------------
Mini Assignment 1 - Task 5
---------------------------------------------
Average age and patient percentage: [Row(average_age=55.267587939698494, count_percent=0.2683)]
